# <font color=#0099CC>**Notebook 3: Implementación de la Estrategia Momentum**</font>

En este notebook se implementa la lógica de inversión tipo MSCI Momentum:

1. **Universo**: activos que han formado parte del S&P 500 durante los últimos 13 meses (interpretación estricta: los 13 meses consecutivos - criterio de estabilidad y liquidez del universo).
2. **Momentum 12 y 6 meses** con retornos logarítmicos y lag de 1 mes (sin usar el mes actual - evita el look-ahead bias).
3. **Z-scores** transversales por mes para combinar ambas ventanas.
4. **Selección**: los 20 activos con mayor score compuesto y peso objetivo 5 % cada uno.

El resultado se guarda en `../outputs/selected_assets.csv` para el motor de backtest. 

In [10]:
# Librerías
import os
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Permite importar utilidades compartidas desde la raíz del proyecto
project_root = Path.cwd().resolve()
if not (project_root / 'notebook_utils.py').exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from notebook_utils import configure_notebook_display, get_project_paths

# Configuración de visualización
configure_notebook_display(max_columns=None, max_rows=100)

BACKTEST_START = pd.Timestamp('2015-01-01')
BACKTEST_END = pd.Timestamp('2026-01-30')

# Rutas de entrada / salida centralizadas
PATHS = get_project_paths(project_root)
DATASETS_DIR = PATHS.datasets
OUTPUTS_DIR = PATHS.outputs

> <u>Nota</u>: Se aplica una arquitectura reutilizable basada en el módulo `notebook_utils.py` (rutas y configuración comunes). Esto reduce acoplamiento entre notebooks, mejora mantenibilidad y ayuda a mantener consistencia de estilo y documentación (PEP 8).

## <font color=#0099CC>**1. CARGA DE DATOS MENSUALES**</font>

### <font color=#336699>**1.1. Retornos logarítmicos y precios mensuales**</font>

Se cargan los retornos logarítmicos mensuales y los precios (open/close) al último día hábil de cada mes, generados en el Notebook 2, junto con el histórico del S&P 500 para construir el universo.

In [11]:
# Carga de datos mensuales y del histórico S&P 500

# Precios y retornos mensuales preparados en el Notebook 2
log_rets = pd.read_parquet(DATASETS_DIR / 'log_returns_monthly.parquet')
monthly_close = pd.read_parquet(DATASETS_DIR / 'monthly_close_prices.parquet')
monthly_open = pd.read_parquet(DATASETS_DIR / 'monthly_open_prices.parquet')

# Histórico completo con in_sp500 para construir el universo
sp500_hist = pd.read_parquet(
    DATASETS_DIR / 'sp500_history_deduplicated.parquet'
)  # histórico deduplicado (N2)
sp500_hist['date'] = pd.to_datetime(sp500_hist['date'])
sp500_hist = sp500_hist.set_index('date').sort_index()

print('> Datos cargados:')
print(f"  - log_returns_monthly: {log_rets.shape}")
print(f"  - monthly_close: {monthly_close.shape}")
print(f"  - monthly_open: {monthly_open.shape}")
print(f"  - sp500_history: {sp500_hist.shape}")

> Datos cargados:
  - log_returns_monthly: (146, 1234)
  - monthly_close: (147, 1234)
  - monthly_open: (147, 1234)
  - sp500_history: (7043936, 13)


> <u>Comentario</u>: Las dimensiones de los Parquet son coherentes: una fila por mes en retornos y precios mensuales, y el histórico S&P 500 en formato largo (fecha × símbolo) para el pivot de pertenencia al índice.

## <font color=#0099CC>**2. UNIVERSO DE INVERSIÓN (13 MESES)**</font>

### <font color=#336699>**2.1. Definición del universo**</font>

El universo abarca todos aquellos activos que han formado parte del S&P 500 durante los últimos 13 meses. Aquí se interpreta de forma **estricta**: un activo entra en el universo en el mes (t) solo si ha estado en el índice en **todos** los meses de la ventana (t-13 a t-1). Así se evita incluir entradas recientes con historial corto, introduciendo un criterio de estabilidad y liquidez del universo.

Para que en la primera fecha de rebalanceo (fin de enero 2015) esta ventana esté completa, el histórico de pertenencia al índice (in_sp500) no se recorta al periodo de backtest: se utiliza la serie completa del dataset del S&P 500, de modo que en cada mes haya 13 meses de historia disponibles.

In [12]:
# Universo de inversión: activos en el S&P 500 durante los últimos 13 meses

# Pivot diario de in_sp500 (0/1) por símbolo
in_sp500_daily = sp500_hist.pivot_table(
    index=sp500_hist.index,
    columns='symbol',
    values='in_sp500',
    aggfunc='last'
).astype(float)

# Pasar a frecuencia mensual (último valor de cada mes)
in_sp500_monthly = in_sp500_daily.resample('ME').last()

# Asegurar alineación temporal con los retornos mensuales: evitamos look-ahead / desfases silenciosos
# Si no hay info de que perteneciera al índice (NaN): no pertenecía al índice
in_sp500_monthly = in_sp500_monthly.reindex(log_rets.index).fillna(0.0)

# Universo: activos que han estado en el índice en TODOS los 13 meses anteriores al mes de rebalanceo (t-13 a t-1)
# Rolling ventana 13 meses; shift(1) para que en la fecha t la elegibilidad dependa solo de t-13..t-1 (sin incluir t), evitando look-ahead.
window = 13
membership_13m = (
    in_sp500_monthly
    .rolling(window=window, min_periods=window)
    .sum()
)

# Un activo pertenece al universo en t si ha estado en el S&P 500 en los 13 meses anteriores al actual (t-13 a t-1)
universe_mask = (membership_13m.shift(1) == window)

print('> Universo 13 meses calculado')
print(f"  - Shape universe_mask: {universe_mask.shape}")

> Universo 13 meses calculado
  - Shape universe_mask: (146, 1234)


## <font color=#0099CC>**3. MOMENTUM 12 Y 6 MESES (CON LAG)**</font>

### <font color=#336699>**3.1. Cálculo de R_12 y R_6**</font>

Con retornos logarítmicos mensuales:

$$
r_t = \ln\left(\frac{P_t}{P_{t-1}}\right)
$$

se definen:

- **R_12 (momentum 12 meses)**: rentabilidad desde el mes \(t-13\) al mes \(t-1\),

$$
R_{12,t} = \sum_{k=1}^{12} r_{t-k}
$$

- **R_6 (momentum 6 meses)**: rentabilidad desde el mes \(t-7\) al mes \(t-1\),

$$
R_{6,t} = \sum_{k=1}^{6} r_{t-k}
$$

El **lag de 1 mes** se implementa con `.shift(1)`: para el rebalanceo en la fecha \(t\) solo se usan retornos hasta \(t-1\), evitando *look-ahead bias* (no se utiliza el mes actual).

In [13]:
# Cálculo de momentum 12 y 6 meses con lag de 1 mes

# log_rets.index son fines de mes. r_t = log(P_t / P_{t-1}).
# Para el rebalanceo en el mes t usamos retornos hasta t-1.

mom_12 = (
    log_rets
    .rolling(window=12, min_periods=12)
    .sum()
    .shift(1)
)

mom_6 = (
    log_rets
    .rolling(window=6, min_periods=6)
    .sum()
    .shift(1)
)

# Filtrar al periodo de backtest
mask_dates = (log_rets.index >= BACKTEST_START) & (log_rets.index <= BACKTEST_END)

mom_12 = mom_12.loc[mask_dates]
mom_6 = mom_6.loc[mask_dates]
universe_mask_bt = universe_mask.loc[mask_dates]

print('> Momentum 12 y 6 meses calculado (con lag)')
print(f"\nmom_12: {mom_12.shape}")
display(mom_12.head())
print(f"\nmom_6: {mom_6.shape}")
display(mom_6.head())

> Momentum 12 y 6 meses calculado (con lag)

mom_12: (132, 1234)


symbol,A,AABA-201910,AAL,AAMRQ-201312,AAP,AAPL,AAV-199901,ABBV,ABI-200811,ABKFQ-201304,ABMD-202212,ABNB,ABS-200606,ABT,ACAS-201701,ACGL,ACKH-200712,ACN,ACS-201002,ACV-201105,ACY-199411,ADBE,ADCT-201012,ADI,ADM,ADP,ADSK,ADT-201604,AEE,AEP,AES,AET-201811,AFL,AFS-200011,AGC-200108,AGN-201503,AHM-199810,AIG,AIT-199910,AIV,AIZ,AJG,AKAM,AKS-202003,AL-200711,ALB,ALGN,ALK,ALL,ALLE,ALTR-201512,ALXN-202107,AM-201308,AMAT,AMCC-201701,AMCR,AMD,AME,AMG,AMGN,AMH-199709,AMP,AMT,AMTM,AMX-199311,AMZN,AN-199812,ANDV-201809,ANDW-200712,ANET,ANF,ANRZQ-201607,ANSS-202507,ANV-199904,AON,AOS,APA,APC-201908,APCC-200702,APD,APH,APO,APOL-201702,APP,APTV,AR-199911,ARC-200004,ARE,ARES,ARG-201605,AS-199909,ASC-199906,ASH,ASN-200710,ASND-199906,ASO-200611,AT-200711,ATGE,ATI-199906,ATO,ATVI-202310,AV-200710,AVB,AVGO,AVP-202001,AVY,AW-200812,AWE-200410,AWK,AXON,AXP,AYE-201102,AYI,AZA-200106,AZO,B,BA,BAC,BALL,BAX,BAY-199808,BBBYQ-202309,BBI-199801,BBWI,BBY,BC,BCO,BCR-201712,BDK-201003,BDX,BEAM-201404,BEN,BEV-200603,BF.B,BFH,BFI-199907,BFO-200010,BG,BGEN-200311,BGGSQ-202101,BHF,BHMSQ-200401,BIGGQ,BIO,BJS-201004,BK,BKB-199909,BKNG,BKR,BLDR,BLK,BLS-200612,BLY-199612,BMC-201309,BMET-200709,BMG-200101,BMS-201906,BMY,BN-199503,BNI-201002,BNL-199806,BNSSA-201206,BOAT-199701,BOL-200710,BR-200603,BRCM-201601,BRK.B,BRL-200812,BRNO-199508,BRO,BSC-200805,BSET,BSX,BT-199906,BTUUQ-201704,BUD-200811,BV-199409,BVSN-202005,BWA,BWY-199510,BX,BXLT-201606,BXP,C,CA-201811,CAG,CAH,CAL,CAM-201604,CAR,CARR,CAT,CB-201601,CBB-199801,CBE-201211,CBH-200803,CBL-199010,CBOE,CBRE,CBS-200005,CBSS-200709,CCB-199602,CCEP,CCI,CCK,CCL,CCTYQ-201109,CCU-200807,CDNS,CDW,CE-200402,CEG,CELG-201911,CEN-200711,CEPH-201110,CERN-202206,CF,CFC-200806,CFG,CFL-199804,CFN-201503,CG-200011,CGP-200101,CHA-200006,CHD,CHIR-200604,CHKAQ-202102,CHRS-201206,CHRW,CHTR,CI,CIC-199505,CIEN,CIN-200603,CINF,CITGQ-200912,CKL-199505,CL,CLF,CLX,CMA,CMB-199603,CMCSA,CMCSK-201512,CME,CMG,CMI,CMS,CMVT-201302,CMX-200703,CNA,CNC,CNCEQ-200309,CNG-200001,CNP,CNW-201510,CNX,CNXT-201104,COC-200208,COF,COIN,COL-201811,COMS-201004,COO,COP,COR,COST,COTY,COV-201501,CPAY,CPB,CPGX-201606,CPNLQ-200802,CPPRQ-202102,CPQ-200205,CPRI,CPRT,CPT,CPWR-201412,CRH,CRL,CRM,CRR-199706,CRWD,CSCO,CSE-199911,CSGP,CSR-200006,CSRA-201804,CSX,CTAS,CTB-202106,CTCO-199110,CTLT-202412,CTRA,CTSH,CTVA,CTX-200908,CTXS-202209,CUE-199309,CVC-201606,CVG-201810,CVGQE-200412,CVH-201305,CVN-199112,CVNA,CVS,CVX,CXO-202101,CXT,CYM-199912,CYR-199606,CZR,D,DAL,DASH,DAY,DCNAQ-200801,DD,DDOG,DDS,DE,DEC-199806,DECK,DELL-201310,DFODQ-202106,DFS-202505,DG,DGN-199910,DGX,DHI,DHR,DI-199809,DIGI-199809,DINO,DIS,DISCK-202204,DISH-202312,DJ-200712,DLR,DLTR,DLX,DNB-201902,DNRCQ-202009,DOC,DOFSQ-202104,DOV,DOW-201708,DPHIQ-200910,DPZ,DRE-202209,DRI,DTE,DTV-201507,DUK,DVA,DVN,DWD-199705,DXC,DXCM,DYHGQ-200012,DYN-201804,E-199504,EA,EBAY,EC-200606,ECH-199807,ECL,ECO-200301,ED,EDS-200808,EFU-200011,EFX,EG,EHC,EIX,EKDKQ-201309,EL,ELV,EMC-201609,EME,EMN,EMR,ENDPQ-202405,ENPH,ENRNQ-200411,ENS-199708,EOG,EOP-200702,EP-201205,EPAM,EQ-200906,EQIX,EQR,EQT,ERIE,ES,ESRX-201812,ESS,ESY-199505,ETFC-202010,ETN,ETR,ETS-200603,ETSY,EVHC-201810,EVRG,EW,EXC,EXE,EXPD,EXPE,EXR,F,FANG,FAST,FBF-200403,FBIN,FBO-199603,FCN-199810,FCX,FDC-200709,FDO-201507,FDS,FDX,FE,FFB-199512,FFIV,FG-199804,FHI,FHN,FICO,FIS,FISV,FITB,FIX,FJ-200011,FJCC-200809,FL-202509,FLIR-202105,FLMIQ-200408,FLR,FLS,FLTWQ-200907,FMC,FMCC,FMY-199905,FNB-199511,FNMA,FOSL,FOXA,FPC-200011,FRCB,FRM-199612,FRO-199909,FRT,FRX-201406,FSH-200611,FSL.B-200612,FSLR,FTI,FTL-200004,FTNT,FTRCQ-202104,FTV,FWLT-201412,G-200509,GAP,GAPTQ-201203,GAS-201606,GBLXQ-200312,GCO,GD,GDDY,GDT-200604,GDW-200609,GE,GEB-199408,GEHC,GEN,GENZ-201104,GEV,GFS.A-199810,GGP-201808,GHC-199308,GIC-200001,GIDL-199709,GILD,GIS,GL,GLD-199705,GLK-200507,GLW,GM,GMCR-201603,GME,GNE-199906,GNN-199006,GNRC,GNT-199806,GNW,GOOGL,GOSHA-200507,GP-200512,GPC,GPN,GPU-200111,GQ-199404,GR-201207,GRA-202109,GRMN,GRN-199812,GS,GSX-199810,GT,GTE-200006,GTW-200710,GWF-199707,GWOW-201009,GWW,H-20


mom_6: (132, 1234)


symbol,A,AABA-201910,AAL,AAMRQ-201312,AAP,AAPL,AAV-199901,ABBV,ABI-200811,ABKFQ-201304,ABMD-202212,ABNB,ABS-200606,ABT,ACAS-201701,ACGL,ACKH-200712,ACN,ACS-201002,ACV-201105,ACY-199411,ADBE,ADCT-201012,ADI,ADM,ADP,ADSK,ADT-201604,AEE,AEP,AES,AET-201811,AFL,AFS-200011,AGC-200108,AGN-201503,AHM-199810,AIG,AIT-199910,AIV,AIZ,AJG,AKAM,AKS-202003,AL-200711,ALB,ALGN,ALK,ALL,ALLE,ALTR-201512,ALXN-202107,AM-201308,AMAT,AMCC-201701,AMCR,AMD,AME,AMG,AMGN,AMH-199709,AMP,AMT,AMTM,AMX-199311,AMZN,AN-199812,ANDV-201809,ANDW-200712,ANET,ANF,ANRZQ-201607,ANSS-202507,ANV-199904,AON,AOS,APA,APC-201908,APCC-200702,APD,APH,APO,APOL-201702,APP,APTV,AR-199911,ARC-200004,ARE,ARES,ARG-201605,AS-199909,ASC-199906,ASH,ASN-200710,ASND-199906,ASO-200611,AT-200711,ATGE,ATI-199906,ATO,ATVI-202310,AV-200710,AVB,AVGO,AVP-202001,AVY,AW-200812,AWE-200410,AWK,AXON,AXP,AYE-201102,AYI,AZA-200106,AZO,B,BA,BAC,BALL,BAX,BAY-199808,BBBYQ-202309,BBI-199801,BBWI,BBY,BC,BCO,BCR-201712,BDK-201003,BDX,BEAM-201404,BEN,BEV-200603,BF.B,BFH,BFI-199907,BFO-200010,BG,BGEN-200311,BGGSQ-202101,BHF,BHMSQ-200401,BIGGQ,BIO,BJS-201004,BK,BKB-199909,BKNG,BKR,BLDR,BLK,BLS-200612,BLY-199612,BMC-201309,BMET-200709,BMG-200101,BMS-201906,BMY,BN-199503,BNI-201002,BNL-199806,BNSSA-201206,BOAT-199701,BOL-200710,BR-200603,BRCM-201601,BRK.B,BRL-200812,BRNO-199508,BRO,BSC-200805,BSET,BSX,BT-199906,BTUUQ-201704,BUD-200811,BV-199409,BVSN-202005,BWA,BWY-199510,BX,BXLT-201606,BXP,C,CA-201811,CAG,CAH,CAL,CAM-201604,CAR,CARR,CAT,CB-201601,CBB-199801,CBE-201211,CBH-200803,CBL-199010,CBOE,CBRE,CBS-200005,CBSS-200709,CCB-199602,CCEP,CCI,CCK,CCL,CCTYQ-201109,CCU-200807,CDNS,CDW,CE-200402,CEG,CELG-201911,CEN-200711,CEPH-201110,CERN-202206,CF,CFC-200806,CFG,CFL-199804,CFN-201503,CG-200011,CGP-200101,CHA-200006,CHD,CHIR-200604,CHKAQ-202102,CHRS-201206,CHRW,CHTR,CI,CIC-199505,CIEN,CIN-200603,CINF,CITGQ-200912,CKL-199505,CL,CLF,CLX,CMA,CMB-199603,CMCSA,CMCSK-201512,CME,CMG,CMI,CMS,CMVT-201302,CMX-200703,CNA,CNC,CNCEQ-200309,CNG-200001,CNP,CNW-201510,CNX,CNXT-201104,COC-200208,COF,COIN,COL-201811,COMS-201004,COO,COP,COR,COST,COTY,COV-201501,CPAY,CPB,CPGX-201606,CPNLQ-200802,CPPRQ-202102,CPQ-200205,CPRI,CPRT,CPT,CPWR-201412,CRH,CRL,CRM,CRR-199706,CRWD,CSCO,CSE-199911,CSGP,CSR-200006,CSRA-201804,CSX,CTAS,CTB-202106,CTCO-199110,CTLT-202412,CTRA,CTSH,CTVA,CTX-200908,CTXS-202209,CUE-199309,CVC-201606,CVG-201810,CVGQE-200412,CVH-201305,CVN-199112,CVNA,CVS,CVX,CXO-202101,CXT,CYM-199912,CYR-199606,CZR,D,DAL,DASH,DAY,DCNAQ-200801,DD,DDOG,DDS,DE,DEC-199806,DECK,DELL-201310,DFODQ-202106,DFS-202505,DG,DGN-199910,DGX,DHI,DHR,DI-199809,DIGI-199809,DINO,DIS,DISCK-202204,DISH-202312,DJ-200712,DLR,DLTR,DLX,DNB-201902,DNRCQ-202009,DOC,DOFSQ-202104,DOV,DOW-201708,DPHIQ-200910,DPZ,DRE-202209,DRI,DTE,DTV-201507,DUK,DVA,DVN,DWD-199705,DXC,DXCM,DYHGQ-200012,DYN-201804,E-199504,EA,EBAY,EC-200606,ECH-199807,ECL,ECO-200301,ED,EDS-200808,EFU-200011,EFX,EG,EHC,EIX,EKDKQ-201309,EL,ELV,EMC-201609,EME,EMN,EMR,ENDPQ-202405,ENPH,ENRNQ-200411,ENS-199708,EOG,EOP-200702,EP-201205,EPAM,EQ-200906,EQIX,EQR,EQT,ERIE,ES,ESRX-201812,ESS,ESY-199505,ETFC-202010,ETN,ETR,ETS-200603,ETSY,EVHC-201810,EVRG,EW,EXC,EXE,EXPD,EXPE,EXR,F,FANG,FAST,FBF-200403,FBIN,FBO-199603,FCN-199810,FCX,FDC-200709,FDO-201507,FDS,FDX,FE,FFB-199512,FFIV,FG-199804,FHI,FHN,FICO,FIS,FISV,FITB,FIX,FJ-200011,FJCC-200809,FL-202509,FLIR-202105,FLMIQ-200408,FLR,FLS,FLTWQ-200907,FMC,FMCC,FMY-199905,FNB-199511,FNMA,FOSL,FOXA,FPC-200011,FRCB,FRM-199612,FRO-199909,FRT,FRX-201406,FSH-200611,FSL.B-200612,FSLR,FTI,FTL-200004,FTNT,FTRCQ-202104,FTV,FWLT-201412,G-200509,GAP,GAPTQ-201203,GAS-201606,GBLXQ-200312,GCO,GD,GDDY,GDT-200604,GDW-200609,GE,GEB-199408,GEHC,GEN,GENZ-201104,GEV,GFS.A-199810,GGP-201808,GHC-199308,GIC-200001,GIDL-199709,GILD,GIS,GL,GLD-199705,GLK-200507,GLW,GM,GMCR-201603,GME,GNE-199906,GNN-199006,GNRC,GNT-199806,GNW,GOOGL,GOSHA-200507,GP-200512,GPC,GPN,GPU-200111,GQ-199404,GR-201207,GRA-202109,GRMN,GRN-199812,GS,GSX-199810,GT,GTE-200006,GTW-200710,GWF-199707,GWOW-201009,GWW,H-20

> <u>Comentario</u>: Tras filtrar al periodo de backtest (desde 2015-01-01), el número de meses con señal válida es coherente con 12 y 6 meses de ventana más el shift.

## <font color=#0099CC>**4. Z-SCORES Y SCORE COMPUESTO**</font>

### <font color=#336699>**4.1. Estandarización transversal**</font>

Las volatilidades de las ventanas de 6 y 12 meses no son directamente comparables en nivel, por lo que se estandarizan los retornos **dentro del universo de cada mes**.

La estandarización transversal se define como:

$$
Z_{i,t} = \frac{x_{i,t} - \mu_t}{\sigma_t}
$$

donde:

- x(i,t) es el valor del activo (i) en la fecha (t),
- mu(t) es la media transversal en la fecha (t),
- sigma(t) es la desviación típica transversal en la fecha (t).

Se calculan así:

$$
Z_{6,i,t} = \frac{R_{6,i,t} - \mu_{6,t}}{\sigma_{6,t}}
$$

$$
Z_{12,i,t} = \frac{R_{12,i,t} - \mu_{12,t}}{\sigma_{12,t}}
$$

El **score final compuesto** se define como la media simple de ambos z-scores:

$$
Score_{i,t} = \frac{Z_{6,i,t} + Z_{12,i,t}}{2}
$$

Esta construcción sigue la metodología utilizada en índices de momentum como los de MSCI.

In [14]:
# Cálculo de Z-scores y score compuesto por mes

# Aplicar máscara de universo (fuera del universo => NaN)
mom_12_u = mom_12.where(universe_mask_bt)
mom_6_u = mom_6.where(universe_mask_bt)

# Medias y desviaciones transversales (por fecha)
mean_12 = mom_12_u.mean(axis=1)
std_12 = mom_12_u.std(axis=1, ddof=0)
mean_6 = mom_6_u.mean(axis=1)
std_6 = mom_6_u.std(axis=1, ddof=0)

# Evitar divisiones por cero
std_12 = std_12.replace(0, np.nan)
std_6 = std_6.replace(0, np.nan)

z_12 = (mom_12_u.sub(mean_12, axis=0)).div(std_12, axis=0)
z_6 = (mom_6_u.sub(mean_6, axis=0)).div(std_6, axis=0)

score = (z_12 + z_6) / 2.0

print('> Z-scores y score compuesto calculados')
display(score.head())

> Z-scores y score compuesto calculados


symbol,A,AABA-201910,AAL,AAMRQ-201312,AAP,AAPL,AAV-199901,ABBV,ABI-200811,ABKFQ-201304,ABMD-202212,ABNB,ABS-200606,ABT,ACAS-201701,ACGL,ACKH-200712,ACN,ACS-201002,ACV-201105,ACY-199411,ADBE,ADCT-201012,ADI,ADM,ADP,ADSK,ADT-201604,AEE,AEP,AES,AET-201811,AFL,AFS-200011,AGC-200108,AGN-201503,AHM-199810,AIG,AIT-199910,AIV,AIZ,AJG,AKAM,AKS-202003,AL-200711,ALB,ALGN,ALK,ALL,ALLE,ALTR-201512,ALXN-202107,AM-201308,AMAT,AMCC-201701,AMCR,AMD,AME,AMG,AMGN,AMH-199709,AMP,AMT,AMTM,AMX-199311,AMZN,AN-199812,ANDV-201809,ANDW-200712,ANET,ANF,ANRZQ-201607,ANSS-202507,ANV-199904,AON,AOS,APA,APC-201908,APCC-200702,APD,APH,APO,APOL-201702,APP,APTV,AR-199911,ARC-200004,ARE,ARES,ARG-201605,AS-199909,ASC-199906,ASH,ASN-200710,ASND-199906,ASO-200611,AT-200711,ATGE,ATI-199906,ATO,ATVI-202310,AV-200710,AVB,AVGO,AVP-202001,AVY,AW-200812,AWE-200410,AWK,AXON,AXP,AYE-201102,AYI,AZA-200106,AZO,B,BA,BAC,BALL,BAX,BAY-199808,BBBYQ-202309,BBI-199801,BBWI,BBY,BC,BCO,BCR-201712,BDK-201003,BDX,BEAM-201404,BEN,BEV-200603,BF.B,BFH,BFI-199907,BFO-200010,BG,BGEN-200311,BGGSQ-202101,BHF,BHMSQ-200401,BIGGQ,BIO,BJS-201004,BK,BKB-199909,BKNG,BKR,BLDR,BLK,BLS-200612,BLY-199612,BMC-201309,BMET-200709,BMG-200101,BMS-201906,BMY,BN-199503,BNI-201002,BNL-199806,BNSSA-201206,BOAT-199701,BOL-200710,BR-200603,BRCM-201601,BRK.B,BRL-200812,BRNO-199508,BRO,BSC-200805,BSET,BSX,BT-199906,BTUUQ-201704,BUD-200811,BV-199409,BVSN-202005,BWA,BWY-199510,BX,BXLT-201606,BXP,C,CA-201811,CAG,CAH,CAL,CAM-201604,CAR,CARR,CAT,CB-201601,CBB-199801,CBE-201211,CBH-200803,CBL-199010,CBOE,CBRE,CBS-200005,CBSS-200709,CCB-199602,CCEP,CCI,CCK,CCL,CCTYQ-201109,CCU-200807,CDNS,CDW,CE-200402,CEG,CELG-201911,CEN-200711,CEPH-201110,CERN-202206,CF,CFC-200806,CFG,CFL-199804,CFN-201503,CG-200011,CGP-200101,CHA-200006,CHD,CHIR-200604,CHKAQ-202102,CHRS-201206,CHRW,CHTR,CI,CIC-199505,CIEN,CIN-200603,CINF,CITGQ-200912,CKL-199505,CL,CLF,CLX,CMA,CMB-199603,CMCSA,CMCSK-201512,CME,CMG,CMI,CMS,CMVT-201302,CMX-200703,CNA,CNC,CNCEQ-200309,CNG-200001,CNP,CNW-201510,CNX,CNXT-201104,COC-200208,COF,COIN,COL-201811,COMS-201004,COO,COP,COR,COST,COTY,COV-201501,CPAY,CPB,CPGX-201606,CPNLQ-200802,CPPRQ-202102,CPQ-200205,CPRI,CPRT,CPT,CPWR-201412,CRH,CRL,CRM,CRR-199706,CRWD,CSCO,CSE-199911,CSGP,CSR-200006,CSRA-201804,CSX,CTAS,CTB-202106,CTCO-199110,CTLT-202412,CTRA,CTSH,CTVA,CTX-200908,CTXS-202209,CUE-199309,CVC-201606,CVG-201810,CVGQE-200412,CVH-201305,CVN-199112,CVNA,CVS,CVX,CXO-202101,CXT,CYM-199912,CYR-199606,CZR,D,DAL,DASH,DAY,DCNAQ-200801,DD,DDOG,DDS,DE,DEC-199806,DECK,DELL-201310,DFODQ-202106,DFS-202505,DG,DGN-199910,DGX,DHI,DHR,DI-199809,DIGI-199809,DINO,DIS,DISCK-202204,DISH-202312,DJ-200712,DLR,DLTR,DLX,DNB-201902,DNRCQ-202009,DOC,DOFSQ-202104,DOV,DOW-201708,DPHIQ-200910,DPZ,DRE-202209,DRI,DTE,DTV-201507,DUK,DVA,DVN,DWD-199705,DXC,DXCM,DYHGQ-200012,DYN-201804,E-199504,EA,EBAY,EC-200606,ECH-199807,ECL,ECO-200301,ED,EDS-200808,EFU-200011,EFX,EG,EHC,EIX,EKDKQ-201309,EL,ELV,EMC-201609,EME,EMN,EMR,ENDPQ-202405,ENPH,ENRNQ-200411,ENS-199708,EOG,EOP-200702,EP-201205,EPAM,EQ-200906,EQIX,EQR,EQT,ERIE,ES,ESRX-201812,ESS,ESY-199505,ETFC-202010,ETN,ETR,ETS-200603,ETSY,EVHC-201810,EVRG,EW,EXC,EXE,EXPD,EXPE,EXR,F,FANG,FAST,FBF-200403,FBIN,FBO-199603,FCN-199810,FCX,FDC-200709,FDO-201507,FDS,FDX,FE,FFB-199512,FFIV,FG-199804,FHI,FHN,FICO,FIS,FISV,FITB,FIX,FJ-200011,FJCC-200809,FL-202509,FLIR-202105,FLMIQ-200408,FLR,FLS,FLTWQ-200907,FMC,FMCC,FMY-199905,FNB-199511,FNMA,FOSL,FOXA,FPC-200011,FRCB,FRM-199612,FRO-199909,FRT,FRX-201406,FSH-200611,FSL.B-200612,FSLR,FTI,FTL-200004,FTNT,FTRCQ-202104,FTV,FWLT-201412,G-200509,GAP,GAPTQ-201203,GAS-201606,GBLXQ-200312,GCO,GD,GDDY,GDT-200604,GDW-200609,GE,GEB-199408,GEHC,GEN,GENZ-201104,GEV,GFS.A-199810,GGP-201808,GHC-199308,GIC-200001,GIDL-199709,GILD,GIS,GL,GLD-199705,GLK-200507,GLW,GM,GMCR-201603,GME,GNE-199906,GNN-199006,GNRC,GNT-199806,GNW,GOOGL,GOSHA-200507,GP-200512,GPC,GPN,GPU-200111,GQ-199404,GR-201207,GRA-202109,GRMN,GRN-199812,GS,GSX-199810,GT,GTE-200006,GTW-200710,GWF-199707,GWOW-201009,GWW,H-20

## <font color=#0099CC>**5. SELECCIÓN MENSUAL DE LOS 20 ACTIVOS**</font>

### <font color=#336699>**5.1. Top 20 por score y peso 5 %**</font>

Cada mes se ordenan los activos del universo por score (mayor a menor) y se seleccionan los 20 primeros. A cada uno se asigna un peso objetivo del 5 % del capital disponible.

En el motor de backtest (Notebook 4) ese 5 % se aplica de forma aproximada por dos motivos: 
1) Solo se pueden comprar cantidades enteras de acciones, por lo que el valor de cada posición se redondea y no coincide exactamente con el 5 %.
2) Las ventas se ejecutan al precio de apertura y las compras al de cierre del día de rebalanceo, de modo que el capital realmente invertido en cada activo depende de esos precios y se desvía ligeramente del reparto teórico.

In [15]:
# Selección mensual de los 20 activos con mayor score

records = []

for dt, row in score.iterrows():
    # Filtramos NaN (fuera del universo o sin datos suficientes)
    row_valid = row.dropna()
    # Si ningún activo es elegible ese mes, se salta y no hay rebalanceo
    if row_valid.empty:
        continue

    # Ordena los activos de mayor a menor y selecciona los 20 mejores
    top20 = row_valid.nlargest(20)

    # Asigna ranking y pesos
    for rank, (symbol, sc) in enumerate(top20.items(), start=1):
        records.append(
            {
                'rebalance_date': dt,
                'rank': rank,
                'symbol': symbol,
                'score': sc,
                'weight': 0.05  # 5 % por activo seleccionado
            }
        )

# Convierte la lista de diccionarios en una tabla estructurada
selected_df = pd.DataFrame.from_records(records)

# Ordena y limpia
selected_df = selected_df.sort_values(['rebalance_date', 'rank']).reset_index(drop=True)

# Comprobación: ¿siempre hay 20 activos por rebalanceo?
counts = selected_df.groupby('rebalance_date').size()
n_per_date = counts.value_counts().sort_index()
print('> Selección mensual completada')
display(selected_df.tail())
print('> Comprobación nº activos por fecha de rebalanceo:')
print(f'  > Activos por mes: min={counts.min()}, max={counts.max()}, media={counts.mean():.2f}')
if (counts != 20).any():
    fechas_menos_20 = counts[counts != 20]
    print(f'  > Meses con distinto de 20 activos: {len(fechas_menos_20)}')
    display(fechas_menos_20.to_frame(name='n_activos'))
else:
    print('  > Todas las emisiones tienen exactamente 20 activos.')

> Selección mensual completada


,rebalance_date,rank,symbol,score,weight
2635,2025-12-31,16,KLAC,1.873071,0.05
2636,2025-12-31,17,CHRW,1.720457,0.05
2637,2025-12-31,18,IDXX,1.693367,0.05
2638,2025-12-31,19,IVZ,1.626654,0.05
2639,2025-12-31,20,TPR,1.597688,0.05


> Comprobación nº activos por fecha de rebalanceo:
  > Activos por mes: min=20, max=20, media=20.00
  > Todas las emisiones tienen exactamente 20 activos.


> <u>Comentario</u>: La tabla muestra, por cada fecha de rebalanceo, el rango 1–20, el símbolo, el score y el peso. Este CSV es el insumo principal del Notebook 4 para ejecutar el backtest.

## <font color=#0099CC>**6. GUARDAR RESULTADOS**</font>

Se exportan: `selected_assets.csv` a `../outputs/` para el motor de backtest y `universe_mask_13m.parquet` a `../datasets/` como fuente de verdad del universo elegible para análisis posteriores (Monte Carlo en Notebook 5).

In [16]:
# Guardar resultados
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(DATASETS_DIR, exist_ok=True)

# 1) Selección mensual para el motor de backtest
output_path = OUTPUTS_DIR / 'selected_assets.csv'
selected_df.to_csv(output_path, index=False)
print(f"> Archivo selected_assets.csv guardado: {output_path}")

# 2) Universo elegible mensual (fuente de verdad para análisis posteriores)
universe_path = DATASETS_DIR / 'universe_mask_13m.parquet'
universe_mask_bt.astype(bool).to_parquet(universe_path)
print(f"> Archivo universe_mask_13m.parquet guardado: {universe_path}")


> Archivo selected_assets.csv guardado: C:\Users\Javi\Desktop\backtesting\outputs\selected_assets.csv
> Archivo universe_mask_13m.parquet guardado: C:\Users\Javi\Desktop\backtesting\datasets\universe_mask_13m.parquet
